In [80]:
import pandas as pd

In [81]:
train_df = pd.read_csv("/Users/priyanshu.tuli/Desktop/machinehack/hate_speech_identification/Dataset/Train.csv")

In [82]:
train_df.head()

,Text_ID,Data,Response
0,1649,Lunch was surreal .,0
1,4705,It offers certificates of completion for many ...,0
2,3320,No amount of educational funds will drag the p...,0
3,6182,"No, lets stay scattered across the earth and t...",0
4,7594,The two teenagers are charged with assaulting ...,0


In [83]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7492 entries, 0 to 7491
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text_ID   7492 non-null   int64 
 1   Data      7492 non-null   object
 2   Response  7492 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 175.7+ KB


In [84]:
test_df = pd.read_csv("/Users/priyanshu.tuli/Desktop/machinehack/hate_speech_identification/Dataset/Test.csv")

In [85]:
test_df.head()

,Text_ID,Data
0,2882,The cross-dressers and faggots are sure not hi...
1,3933,They ca n't stand toe to toe with the white ma...
2,304,There is an active community here - you just n...
3,3325,"Mate, have you been to Britain ?"
4,2626,They are nothing less than indoctrination cent...


In [86]:
test_df.shape

(3211, 2)

In [87]:
test_df.head()

,Text_ID,Data
0,2882,The cross-dressers and faggots are sure not hi...
1,3933,They ca n't stand toe to toe with the white ma...
2,304,There is an active community here - you just n...
3,3325,"Mate, have you been to Britain ?"
4,2626,They are nothing less than indoctrination cent...


In [88]:
train_df.shape

(7492, 3)

In [89]:
train_df["Response"].value_counts()

Response
0    6642
1     850
Name: count, dtype: int64

In [90]:
train_df["Text_ID"].nunique()

7492

In [91]:
train_df.drop(columns=["Text_ID"], inplace=True)

In [92]:
train_df[["Data", "Response"]].head()

,Data,Response
0,Lunch was surreal .,0
1,It offers certificates of completion for many ...,0
2,No amount of educational funds will drag the p...,0
3,"No, lets stay scattered across the earth and t...",0
4,The two teenagers are charged with assaulting ...,0


In [93]:
train_df[train_df["Response"] == 1].head()

,Data,Response
11,High time they get back to mother Affika and f...,1
13,Though I have had problems at one or two that ...,1
14,I think there is more threat from Russians tha...,1
48,Blacks hate whites .,1
50,"Unfortunately, she will find it very difficult...",1


In [94]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/priyanshu.tuli/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/priyanshu.tuli/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/priyanshu.tuli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/priyanshu.tuli/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [95]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

In [96]:
from tqdm import tqdm
tqdm.pandas()

In [97]:
train_df['Cleaned_Data'] = train_df['Data'].progress_apply(preprocess_text)

100%|██████████| 7492/7492 [00:01<00:00, 6504.24it/s]


In [98]:
test_df['Cleaned_Data'] = test_df['Data'].progress_apply(preprocess_text)

100%|██████████| 3211/3211 [00:00<00:00, 6736.10it/s]


In [99]:
train_df[['Data', 'Cleaned_Data']].head()

,Data,Cleaned_Data
0,Lunch was surreal .,lunch surreal
1,It offers certificates of completion for many ...,offer certificate completion many course
2,No amount of educational funds will drag the p...,amount educational fund drag population sludge
3,"No, lets stay scattered across the earth and t...",let stay scattered across earth try change eve...
4,The two teenagers are charged with assaulting ...,two teenager charged assaulting rose powell la...


In [100]:
from gensim.models import Word2Vec

In [101]:
from sklearn.model_selection import train_test_split

In [102]:
train_texts, val_texts, train_labels, val_labels = train_test_split(train_df['Cleaned_Data'], train_df['Response'], test_size=0.2, random_state=42)

In [103]:
test_texts = test_df['Cleaned_Data']

In [112]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

In [113]:
tokenizer = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

In [114]:
def tokenize_data(texts, labels):
    tokens = tokenizer(texts.values.tolist(), padding=True,return_tensors="pt")
    return tokens["input_ids"], tokens["attention_mask"], torch.tensor(labels.values)

In [115]:
def tokenize_test_data(texts):
    tokens = tokenizer(texts.values.tolist(), padding="max_length",return_tensors="pt")
    return tokens["input_ids"], tokens["attention_mask"]

In [116]:
test_input_ids, test_attention_mask = tokenize_test_data(test_texts)

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`input_ids` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:
train_input_ids, train_attention_mask, train_labels = tokenize_data(train_texts, train_labels)

In [36]:
val_input_ids, val_attention_mask, val_labels = tokenize_data(val_texts, val_labels)

In [37]:
class TextDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "label": self.labels[idx]
        }

In [38]:
train_dataset = TextDataset(train_input_ids, train_attention_mask, train_labels)

In [39]:
val_dataset = TextDataset(val_input_ids, val_attention_mask, val_labels)

In [48]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [45]:
class BertClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, 1)  # Binary classification
        self.sigmoid = nn.Sigmoid()  # Sigmoid for binary classification

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token output
        x = self.dropout(cls_output)
        x = self.fc(x)
        return self.sigmoid(x).squeeze()

In [46]:
model = BertClassifier()
criterion = nn.BCELoss()  # Binary cross-entropy loss
optimizer = optim.Adam(model.parameters(), lr=2e-5)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [52]:
def train_model(model, train_loader, epochs=3):
    model.train()
    for epoch in tqdm(range(epochs)):
        total_loss = 0
        for batch in train_loader:
            input_ids, attention_mask, labels = (
                batch["input_ids"],
                batch["attention_mask"],
                batch["label"]
            )
            labels = labels.to(torch.float32)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader)}")
    
    model.eval()
    for batch in val_loader:
        input_ids, attention_mask, labels = (
            batch["input_ids"],
            batch["attention_mask"],
            batch["label"]
        )
        labels = labels.to(torch.float32)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        
    print(f"Validation Loss = {total_loss / len(val_loader)}")

In [53]:
train_model(model, train_loader, epochs=3)

  0%|          | 0/3 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
test_dataset = TextDataset(test_input_ids, test_attention_mask, torch.zeros(test_input_ids.shape[0]))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

ImportError: 
TFBertForSequenceClassification requires the TensorFlow library but it was not found in your environment.
However, we were able to find a PyTorch installation. PyTorch classes do not begin
with "TF", but are otherwise identically named to our TF classes.
If you want to use PyTorch, please use those classes instead!

If you really do want to use TensorFlow, please follow the instructions on the
installation page https://www.tensorflow.org/install that match your environment.
